# 🎬 Piku vs 3 Hyenas — 30-Second Full Slapstick Cartoon Showdown
### ⚡ 6 Clips (5s Each) • Complete Story • Full Hindi Slapstick Voiceover!

**Runtime Check:** Runs on **Standard CPU** (or T4 GPU) with **0% Load on Local Laptop**!

## 📦 Step 1: Install Dependencies & Download 6 Storyboard Assets + Audio

In [ ]:
!pip install -q gradio_client ffmpeg-python
!apt-get update -qq && apt-get install -y -qq ffmpeg

# Fresh clone of repository to fetch images & master audio
!rm -rf videogen
!git clone https://github.com/yashkumarsahu789/videogen.git

import os
assets_dir = "videogen/video_output/piku_30s_slapstick/assets"
audio_file = "videogen/video_output/piku_30s_slapstick/audio/voiceover_30s_master.mp3"

print("✅ 6 Scene Storyboard Assets:", sorted(os.listdir(assets_dir)))
print("✅ 30s Master Voiceover Exists:", os.path.exists(audio_file))

## 🚀 Step 2: Render All 6 Clips (Cloud DiT Engine with Bulletproof Fallback)

In [ ]:
# @title 🚀 Step 2: Render All 6 Clips (Cloud DiT Engine)
# @markdown **Hugging Face Token (Optional):**
# @markdown Agar aapke paas free token hai toh yahan paste karein (starts with `hf_...`).
# @markdown *Agar nahi hai toh ise khali chhod dein — yeh public free pool se chalega!*
HF_TOKEN = "" # @param {type:"string"}

import os, time, shutil
from pathlib import Path
from gradio_client import Client, handle_file

hf_token_clean = HF_TOKEN.strip() if HF_TOKEN and HF_TOKEN.strip() else None

# Auto-format token if user copied with minor prefix typo
if hf_token_clean:
    if hf_token_clean.startswith("HFAK") or hf_token_clean.startswith("HF"):
        hf_token_clean = "hf_" + hf_token_clean[2:]
    elif not hf_token_clean.startswith("hf_"):
        hf_token_clean = "hf_" + hf_token_clean

client = None
if hf_token_clean:
    try:
        print(f"🔑 Testing Hugging Face Token ({hf_token_clean[:6]}...)... ")
        client = Client("Lightricks/ltx-video-distilled", token=hf_token_clean)
        print("✅ Connected with Dedicated Hugging Face Token!")
    except Exception as e:
        print(f"⚠️ Token could not be authenticated ({e}).")
        print("🚀 Automatically falling back to Free Public Cloud Pool...")
        client = Client("Lightricks/ltx-video-distilled")
        print("✅ Connected to Free Public Cloud Pool!")
else:
    print("🚀 Connecting to Free Public Cloud Pool...")
    client = Client("Lightricks/ltx-video-distilled")
    print("✅ Connected to Cloud GPU!")

assets_path = Path("videogen/video_output/piku_30s_slapstick/assets")
output_dir = Path("output_clips_30s")
output_dir.mkdir(parents=True, exist_ok=True)

clips = [
    ("scene1_chase_leaf_slide.jpg", "clip1.mp4", "Three panicked spotted hyenas sliding fast on giant green leaves downhill wide eyed screaming, cheeky baby monkey laughing"),
    ("scene2_banana_peel_slip.jpg", "clip2.mp4", "Three goofy hyenas slipping wildly on giant yellow banana peels spinning like tops in panic, baby monkey Piku pointing laughing"),
    ("scene3_armadillo_bowling.jpg", "clip3.mp4", "Cheeky baby monkey rolling spiky armadillo bowling ball downhill knocking into hyenas sliding on giant leaf"),
    ("scene4_spiderweb_catapult.jpg", "clip4.mp4", "Three hyenas crashing into glowing giant elastic spiderweb trampoline stretching back and slingshotting them into sky"),
    ("scene5_crocodile_belly_crash.jpg", "clip5.mp4", "Three terrified hyenas falling from sky face-planting on sleeping giant green crocodile squishy belly riverbank"),
    ("scene6_victory_mango_laugh.jpg", "clip6.mp4", "Baby monkey Piku eating juicy mango winking double thumbs up, three hyenas stuck upside down in tree branches dizzy stars")
]

clip_paths = []
for idx, (img_name, out_name, prompt) in enumerate(clips):
    img_file = assets_path / img_name
    out_file = output_dir / out_name
    print(f"\n🎬 Rendering Clip {idx+1}/6: {img_name}...")
    t0 = time.time()
    
    res = None
    for attempt in range(4):
        try:
            res = client.predict(
                prompt=prompt,
                negative_prompt="blurry, low quality, jittery, distorted",
                input_image_filepath=handle_file(str(img_file)),
                input_video_filepath=None,
                height_ui=512,
                width_ui=704,
                mode="image-to-video",
                duration_ui=5.0,
                ui_frames_to_use=9,
                seed_ui=50 + idx,
                randomize_seed=False,
                ui_guidance_scale=1,
                improve_texture_flag=True,
                api_name="/image_to_video"
            )
            break
        except Exception as err:
            if attempt < 3:
                print(f"⏳ Cloud queue busy ({err}). Waiting 15s to auto-retry (Attempt {attempt+2}/4)...")
                time.sleep(15)
            else:
                raise err

    video_res = res[0]
    raw_path = video_res["video"] if isinstance(video_res, dict) and "video" in video_res else video_res
    shutil.copyfile(raw_path, str(out_file))
    clip_paths.append(str(out_file))
    print(f"✓ Completed in {round(time.time() - t0, 1)}s -> {out_name}")

print("\n🎉 ALL 6 CLIPS RENDERED SUCCESSFULLY!")


## 🎶 Step 3: Concat & Merge 30-Second Master Hindi Slapstick Audio

In [ ]:
assert len(clip_paths) == 6, f"Expected 6 clips, found {len(clip_paths)}"

with open("concat_30s.txt", "w") as f:
    for cp in clip_paths:
        f.write(f"file '{os.path.abspath(cp)}'\n")

audio_path = "videogen/video_output/piku_30s_slapstick/audio/voiceover_30s_master.mp3"
final_output = "piku_30s_slapstick_cartoon_master.mp4"

!ffmpeg -y -f concat -safe 0 -i concat_30s.txt -i {audio_path} -c:v libx264 -c:a aac -shortest -pix_fmt yuv420p {final_output}
print(f"\n✅ Final 30-Second Master Cartoon Movie Ready: {final_output}")

## 📥 Step 4: Preview & Download

In [ ]:
from IPython.display import HTML, FileLink
from base64 import b64encode

mp4_bytes = open(final_output, 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4_bytes).decode()

display(HTML(f'''
<div style="background:#1E293B; padding:20px; border-radius:16px; border:2px solid #10B981; max-width:760px; margin:auto; text-align:center;">
    <h3 style="color:#F59E0B; margin-top:0;">🐵 Piku vs 3 Hyenas — 30-Second Slapstick Cartoon Master</h3>
    <video width=704 height=512 controls autoplay loop style="border-radius:12px; width:100%; border:2px solid #334155;">
        <source src="{data_url}" type="video/mp4">
    </video>
</div>
'''))

print("\n⬇️ Download Final MP4:")
display(FileLink(final_output))